# 3aransiya · expanded v2 Colab comparison

Train the GRU encoder–decoder and bidirectional GRU/CTC on the same development corpus. The code is embedded in this notebook. The locked test stays closed.

1. Upload `doda-word-pairs-v2.json` through Colab’s Files panel so it is at `/content/doda-word-pairs-v2.json`.
2. In Colab choose **Runtime → Change runtime type → T4 GPU**, then run all cells.
3. Outputs go to `/content/outputs/`. Download anything you want to keep before the runtime resets. Rerunning in the same runtime resumes from its newest valid local checkpoint.

The default saves after each optimizer step. A full runtime reset clears local files. To resume on a fresh runtime, upload the dataset and restore the saved `outputs/` folder before rerunning.

[Colab runtime limits](https://research.google.com/colaboratory/faq.html)


In [ ]:
from pathlib import Path
CONTENT = Path("/content")
OUTPUTS = CONTENT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)
RUN_NAME = "t4-v2-one-pass"
CANDIDATES = ["gru_encoder_decoder_v1", "bidirectional_gru_ctc_v1"]
EPOCHS = 1
BATCH_SIZE = 128
SEED = 42
SAVE_EVERY = 1
STEPS_THIS_SESSION = None  # e.g. 100 to stop after 100 additional batches per candidate
RUN_SYNTHETIC_TESTS = True


In [ ]:
from base64 import b64decode
from hashlib import sha256
from io import BytesIO
from zipfile import ZipFile

import torch

if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0).split():
    raise RuntimeError("Select a T4 GPU in Colab Runtime → Change runtime type")
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))

EXPECTED_DATA = "0ce61f1d95a6aa9c4932e883e116b82b009d3411149d8e9e6fce852e88a9121e"
LOCAL = Path("/content/3aransiya-pilot")
LOCAL.mkdir(exist_ok=True)
WORD_PAIRS = CONTENT / "doda-word-pairs-v2.json"
if not WORD_PAIRS.is_file():
    raise FileNotFoundError(f"Upload the prepared development artifact to {WORD_PAIRS}")
if sha256(WORD_PAIRS.read_bytes()).hexdigest() != EXPECTED_DATA:
    raise ValueError("The prepared word-pair artifact has the wrong checksum")

# Embedded project source snapshot for this notebook.
source = b64decode(
    '...'
)
assert sha256(source).hexdigest() == "8e3253aaed8b5e51951807d5713e569ab4146c0b329441151b4f324ac37e153e"
with ZipFile(BytesIO(source)) as archive:
    for member in archive.namelist():
        path = LOCAL / member
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(archive.read(member))


## Tests before corpus training

Run the synthetic recovery and bundle tests on CPU. They compare an interrupted run with an uninterrupted run, exercise the fallback after a damaged checkpoint, and check prediction parity and bundle rejection. These tests write only temporary files.


In [ ]:
import os
import subprocess
import sys

if RUN_SYNTHETIC_TESTS:
    for pattern in ("test_training_resume.py", "test_inference_bundle.py"):
        subprocess.run(
            [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern],
            cwd=LOCAL,
            env={**os.environ, "OMP_NUM_THREADS": "1", "CUDA_VISIBLE_DEVICES": ""},
            check=True,
        )


## Train or resume on T4

In [ ]:
import os
import subprocess
import sys

for architecture in CANDIDATES:
    run = OUTPUTS / RUN_NAME / architecture
    command = [
        sys.executable, "-m", "model_pilot.train", str(WORD_PAIRS), str(run),
        "--architecture", architecture, "--dataset-version", "v2", "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE), "--seed", str(SEED), "--save-every", str(SAVE_EVERY),
    ]
    if STEPS_THIS_SESSION is not None:
        command.extend(["--steps-this-session", str(STEPS_THIS_SESSION)])
    subprocess.run(command, cwd=LOCAL, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0"}, check=True)


## Validation on held-out development words

The trainer scores both candidates after each epoch on the same validation occurrences. Reports are available after a partial run, too. Strict scores use the retained reference. Variant-aware scores use the twelve pilot spelling equivalences. Missing predictions count as misses in both columns. The locked release test is separate and is reserved for the final selected model.


In [ ]:
import json

completed = {}
def percent(value):
    return "n/a" if value is None else f"{value:.2%}"

for architecture in CANDIDATES:
    run = OUTPUTS / RUN_NAME / architecture
    epoch_reports = sorted(run.glob("validation-epoch-*.json"))
    print()
    print(architecture)
    if epoch_reports:
        print(f"{'Epoch':>5} {'Slice':<10} {'Strict':>9} {'Variants':>9} {'Missing':>8}")
        for report_path in epoch_reports:
            epoch = int(report_path.stem.rsplit("-", 1)[1])
            slices = json.loads(report_path.read_text())["scores"]["slices"]
            for name in slices:
                scores = slices[name]
                strict = percent(scores["strict_accuracy"])
                variants = percent(scores["variant_accuracy"])
                print(f"{epoch:>5} {name:<10} {strict:>9} {variants:>9} {scores['missing']:>8}")

    report_path = run / "validation.json"
    if not report_path.is_file():
        print(f"{architecture}: final export pending until training completes")
        continue
    report = json.loads(report_path.read_text())
    completed[architecture] = (run, report["checkpoint_sha256"])
    print("Final validation coverage")
    print(f"{'Slice':<10} {'Words':>8} {'Forms':>8} {'Strict':>9} {'Variants':>9} {'Missing':>8}")
    for name, scores in report["scores"]["slices"].items():
        strict = percent(scores["strict_accuracy"])
        variants = percent(scores["variant_accuracy"])
        print(f"{name:<10} {scores['occurrences']:>8} {scores['distinct_forms']:>8} "
              f"{strict:>9} {variants:>9} {scores['missing']:>8}")


## Test the exported bundle

Load each completed bundle via the same local pipe protocol the Fedora app uses. Check that a fixed word produces output and that an overlength word is rejected while the helper stays available. This is an integration smoke test. Use the validation table for development accuracy.


In [ ]:
for architecture, (run, checkpoint_digest) in completed.items():
    bundle = run / f"bundle-{checkpoint_digest[:16]}"
    subprocess.run(
        [sys.executable, "-c", '''
import json
from pathlib import Path
import sys
from scratchpad.process_predictor import PredictorError, ProcessPredictor

bundle = Path(sys.argv[1])
manifest = json.loads((bundle / "manifest.json").read_text())
if manifest["source_checkpoint_sha256"] != sys.argv[2]:
    raise ValueError("inference bundle does not match the validation checkpoint")
helper = ProcessPredictor([sys.executable, "-m", "model_pilot.inference_helper", bundle])
try:
    for word in ("salam", "T7", "t7" * 9, "kanchouf", "bghit", "3lach"):
        try:
            expected = helper.predict(word)
        except PredictorError:
            if helper.closed:
                raise
            continue
        break
    else:
        raise RuntimeError("no successful prediction on the fixed fixture")
    try:
        helper.predict("t" * 19)
    except PredictorError:
        if helper.closed:
            raise
    else:
        raise RuntimeError("helper accepted an overlength word")
    if helper.predict(word) != expected:
        raise RuntimeError("prediction changed after a rejected request")
finally:
    helper.close()
''', str(bundle), checkpoint_digest],
        cwd=LOCAL,
        env={**os.environ, "CUDA_VISIBLE_DEVICES": ""},
        check=True,
    )
    print(f"{architecture}: exported bundle passed · {bundle}")


## Results and restart

Look under `/content/outputs/<RUN_NAME>/<architecture>/`. Download the complete `bundle-…` directory for Fedora. Each epoch report records the exact checkpoint digest. A restart after an interrupted validation step resumes from the completed epoch’s saved weights.

Keep the same code, Python, PyTorch, GPU type, and training settings when resuming. A changed runtime or configuration stops with an error to prevent a silent continuation under different assumptions. The two architectures have independent checkpoints.
